# CUM Series 14 — Deep Research V4 Mathematical Frameworks

Testing three new approaches from pure math:
- **Ruiz equilibration** (zero-cost diagonal pre-conditioning) + fewer NS steps
- **Frame potential** gradient flow (the universal cubic iteration)
- **Polar Express** (different polynomial each step)

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/BrownHujay/Curvature-Unified-Muon.git'
REPO_DIR = '/content/Curvature-Unified-Muon'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned repo')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Pulled latest')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch
import torch.nn as nn
import torch.nn.functional as F
import time, math
import numpy as np

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

assert torch.cuda.is_available(), 'Need GPU!'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
DATA_DIR = 'benchmarks/tier0/data'
DATA_PATH = os.path.join(DATA_DIR, 'tinyshakespeare.txt')
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        DATA_PATH)
with open(DATA_PATH, 'r') as f:
    TEXT = f.read()
print(f'Dataset: {len(TEXT):,} chars')

In [ ]:
from benchmarks.models.micro_gpt import MicroGPT
from cum.newton_schulz import newton_schulz_orthogonalize
from cum.utils import aspect_ratio_scale
from cum import CUM8v1, CUM14v1

MODEL_CFG = dict(d_model=128, n_heads=4, n_layers=4, d_ff=512, ctx_len=256)
BATCH_SIZE = 32
TOTAL_STEPS = 2000
WARMUP_STEPS = 200
EVAL_EVERY = 250
BASE_LR = 0.02
SEED = 42

test_model = MicroGPT(vocab_size=65, **MODEL_CFG)
print(f'Model: {sum(p.numel() for p in test_model.parameters())/1e6:.1f}M params')
del test_model
print('Imports OK')

In [ ]:
class CharDataset:
    def __init__(self, text, ctx_len=256, device='cpu'):
        self.chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
        self.data = torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long, device=device)
        self.ctx_len = ctx_len

    def get_batch(self, batch_size, split='train', train_ratio=0.9):
        n = int(len(self.data) * train_ratio)
        data = self.data[:n] if split == 'train' else self.data[n:]
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=self.data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y

dataset = CharDataset(TEXT, ctx_len=MODEL_CFG['ctx_len'], device=DEVICE)
print(f'Dataset on {DEVICE}: vocab={dataset.vocab_size}')

In [ ]:
class SimpleMuon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, beta1=0.95, ns_steps=5):
        defaults = dict(lr=lr, beta1=beta1, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if len(state) == 0:
                    state['momentum_buffer'] = torch.zeros_like(p)
                m = state['momentum_buffer']
                m.mul_(group['beta1']).add_(p.grad, alpha=1 - group['beta1'])
                u = p.grad + group['beta1'] * m
                orth = newton_schulz_orthogonalize(u, steps=group['ns_steps'])
                scale = math.sqrt(max(1, p.shape[0] / p.shape[1]))
                p.add_(orth, alpha=-group['lr'] * scale)


def get_lr(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


def train_one(name, main_opt, aux_opt, model, dataset,
              total_steps=TOTAL_STEPS, batch_size=BATCH_SIZE,
              warmup_steps=WARMUP_STEPS, base_lr=BASE_LR, eval_every=EVAL_EVERY):
    model.train()
    trajectory = []

    for _ in range(3):
        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)
        loss.backward()
        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    for step in range(total_steps):
        current_lr = get_lr(step, total_steps, warmup_steps, base_lr)
        if main_opt:
            for g in main_opt.param_groups:
                g['lr'] = current_lr

        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)

        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()
        loss.backward()
        if main_opt: main_opt.step()
        aux_opt.step()

        if step % eval_every == 0 or step == total_steps - 1:
            model.eval()
            vl = []
            for _ in range(20):
                vx, vy = dataset.get_batch(batch_size, split='val')
                with torch.no_grad():
                    _, v = model(vx, vy)
                    vl.append(v.item())
            val_loss = np.mean(vl)
            trajectory.append((step, val_loss))
            model.train()
            elapsed = time.perf_counter() - t0
            print(f'  [{name}] step {step}: val_loss={val_loss:.4f} ({elapsed:.0f}s)')

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    model.eval()
    vl = []
    for _ in range(50):
        vx, vy = dataset.get_batch(batch_size, split='val')
        with torch.no_grad():
            _, v = model(vx, vy)
            vl.append(v.item())
    final_val = np.mean(vl)
    return final_val, trajectory, elapsed


print('Training loop ready')

In [ ]:
def make_model_and_opts(dataset, cfg):
    torch.manual_seed(SEED)
    model = MicroGPT(vocab_size=dataset.vocab_size, **MODEL_CFG).to(DEVICE)
    model = torch.compile(model, mode='reduce-overhead')

    hidden_2d = model.get_hidden_2d_params()
    other = model.get_other_params()
    aux_opt = torch.optim.AdamW(other, lr=3e-4, weight_decay=0.01)

    t = cfg['type']
    if t == 'Muon':
        main_opt = SimpleMuon(hidden_2d, lr=BASE_LR, ns_steps=5)
    elif t == '8v1':
        main_opt = CUM8v1(
            hidden_2d, lr=BASE_LR,
            method='matrix', mode='combined',
            ns_steps=5, save_at=2, blend=0.15,
            input_blend_beta=0.5, input_blend_alpha=0.15,
            total_steps=TOTAL_STEPS,
        )
    elif t == '14v1':
        main_opt = CUM14v1(
            hidden_2d, lr=BASE_LR,
            mode=cfg['mode'],
            ruiz_steps=cfg.get('ruiz_steps', 5),
            ns_steps=cfg.get('ns_steps', 3),
            frame_steps=cfg.get('frame_steps', 7),
            frame_eta=cfg.get('frame_eta', 2.5),
            frame_c=cfg.get('frame_c', 0.88),
            pe_steps=cfg.get('pe_steps', 5),
            td_lambda=cfg.get('td_lambda', 0.5),
            input_blend_beta=0.5, input_blend_alpha=0.15,
        )
    else:
        raise ValueError(f'Unknown: {t}')

    return model, main_opt, aux_opt


def run_all(configs):
    results = []
    for i, cfg in enumerate(configs):
        name = cfg['name']
        print(f'\n{"="*60}')
        print(f'[{i+1}/{len(configs)}] {name}')
        print(f'{"="*60}')
        try:
            model, main_opt, aux_opt = make_model_and_opts(dataset, cfg)
            val_loss, traj, elapsed = train_one(name, main_opt, aux_opt, model, dataset)
            results.append(dict(name=name, type=cfg['type'], val_loss=val_loss,
                                trajectory=traj, elapsed=elapsed, error=None))
            print(f'  FINAL: {name} -> {val_loss:.4f} ({elapsed:.1f}s)')
        except Exception as e:
            import traceback; traceback.print_exc()
            results.append(dict(name=name, type=cfg['type'], val_loss=float('inf'),
                                trajectory=[], elapsed=0, error=str(e)))
        torch.cuda.empty_cache()
    return results


def show_results(results, series='14a'):
    muon = next((r['val_loss'] for r in results if r['type'] == 'Muon'), None)
    combined = next((r['val_loss'] for r in results
                     if r.get('name', '').startswith('8v1')), None)

    print(f'\n## Series {series} Results\n')
    print(f'| Config | Val Loss | vs Muon | vs Combined | Time |')
    print(f'|--------|----------|---------|-------------|------|')
    for r in sorted(results, key=lambda x: x['val_loss']):
        if r.get('error'):
            print(f'| {r["name"]} | FAILED | -- | -- | {r["error"][:30]} |')
            continue
        vm = f'{r["val_loss"] - muon:+.4f}' if muon else '--'
        vc = f'{r["val_loss"] - combined:+.4f}' if combined else '--'
        print(f'| {r["name"]} | {r["val_loss"]:.4f} | {vm} | {vc} | {r["elapsed"]:.0f}s |')


print('Factory ready')

---
## 14a: Ruiz Pre-conditioning + Reduced NS

Can cheap diagonal scaling let us cut NS from 5→3 steps?

In [ ]:
CONFIGS_14A = [
    {'name': 'Muon NS=5', 'type': 'Muon'},
    {'name': '8v1 combined', 'type': '8v1'},
    {'name': 'Ruiz5+NS3 basic', 'type': '14v1', 'mode': 'ruiz_ns',
     'ruiz_steps': 5, 'ns_steps': 3},
    {'name': 'Ruiz5+NS5 basic', 'type': '14v1', 'mode': 'ruiz_ns',
     'ruiz_steps': 5, 'ns_steps': 5},
]

results_14a = run_all(CONFIGS_14A)
show_results(results_14a, '14a: Ruiz Pre-conditioning')

---
## 14b: Frame Potential + Polar Express

Frame potential: the universal cubic σ→σ(1−η(σ²−c²)) with aggressive η + blending.
Polar Express: different polynomial each step + blending.

In [ ]:
CONFIGS_14B = [
    {'name': 'Muon NS=5', 'type': 'Muon'},
    {'name': '8v1 combined', 'type': '8v1'},
    {'name': 'Frame η=2.5 7step', 'type': '14v1', 'mode': 'frame',
     'frame_steps': 7, 'frame_eta': 2.5, 'frame_c': 0.88, 'td_lambda': 0.5},
    {'name': 'Polar Express 5step', 'type': '14v1', 'mode': 'polar_express',
     'pe_steps': 5, 'td_lambda': 0.5},
]

results_14b = run_all(CONFIGS_14B)
show_results(results_14b, '14b: Frame Potential + Polar Express')

---
## 14c: Best of above + combined blending (if anything showed promise)

If Ruiz helped, try Ruiz+NS3 with full TD(λ)+temporal recipe.

In [ ]:
CONFIGS_14C = [
    {'name': 'Muon NS=5', 'type': 'Muon'},
    {'name': '8v1 combined', 'type': '8v1'},
    {'name': 'Ruiz5+NS3 combined', 'type': '14v1', 'mode': 'ruiz_ns_combined',
     'ruiz_steps': 5, 'ns_steps': 3, 'td_lambda': 0.5},
    {'name': 'Ruiz5+NS5 combined', 'type': '14v1', 'mode': 'ruiz_ns_combined',
     'ruiz_steps': 5, 'ns_steps': 5, 'td_lambda': 0.5},
]

results_14c = run_all(CONFIGS_14C)
show_results(results_14c, '14c: Ruiz + Combined Blending')

---
## 14d: Remez-Optimal Polar Express

Step-adaptive coefficients computed via Remez algorithm (differential_evolution).
Each step's polynomial is minimax-optimal for its spectral interval, targeting σ=0.88.
Step 1 is MUCH more aggressive (a=5.96) than standard NS (a=3.44).

In [ ]:
CONFIGS_14D = [
    {'name': 'Muon NS=5', 'type': 'Muon'},
    {'name': '8v1 combined', 'type': '8v1'},
    {'name': 'Polar Remez 5step', 'type': '14v1', 'mode': 'polar_express',
     'pe_steps': 5, 'td_lambda': 0.5},
    {'name': 'Polar Remez 3step', 'type': '14v1', 'mode': 'polar_express',
     'pe_steps': 3, 'td_lambda': 0.5},
]

results_14d = run_all(CONFIGS_14D)
show_results(results_14d, '14d: Remez-Optimal Polar Express')